# LLM-as-a-Judge: Experimental Results Analysis
This notebook visualizes the accuracy of various models across different prompting strategies and debiasing techniques.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import os

# Load data
df = pd.read_csv("summary_results.csv")
df["accuracy_pct"] = df["accuracy"] * 100

# Create a readable configuration string
def get_config(row):
    configs = []
    if row["pos_debias"]:
        configs.append("PosDebias")
    if row["self_consist"]:
        configs.append("SelfConsist")
    return " + ".join(configs) if configs else "Baseline"

df["improvement_strategy"] = df.apply(get_config, axis=1)
print("✅ Data loaded. Found experiments for:", df["model"].unique())
df.head(100)

## 1. Baseline Performance Comparison
Comparing model accuracy across different prompt types (Zero-shot, Few-shot, CoT) without improvements.

In [ ]:
plt.figure(figsize=(12, 6))
baseline_df = df[(df["improvement_strategy"] == "Baseline") & (df["prompt_type"].isin(["zero_shot", "cot"]))]

ax = sns.barplot(
    data=baseline_df, 
    x="model", 
    y="accuracy_pct", 
    hue="prompt_type",
    palette="viridis"
)

# Add accuracy labels on top of bars
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.title("Baseline Accuracy by Model and Prompt Type")
plt.ylabel("Accuracy (%)")
plt.xlabel("Model")
plt.xticks(rotation=45)
plt.legend(title="Prompt Type", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

## 2. Impact of Strategy Improvements
Comparing the baseline performance against Position Debiasing and Self-Consistency strategies.

In [ ]:
if len(df["improvement_strategy"].unique()) > 1:
	plt.figure(figsize=(12, 6))
	ax = sns.barplot(
		data=df, 
		x="prompt_type", 
		y="accuracy_pct", 
		hue="improvement_strategy",
		palette="Set2"
)
	
	# Add accuracy labels on top of bars
	for container in ax.containers:
		ax.bar_label(container, fmt='%.1f%%', padding=3)

	plt.title("Impact of Improvements Across Prompt Types (Averaged Over Models)")
	plt.ylabel("Accuracy (%)")
	plt.xlabel("Prompt Type")
	plt.legend(title="Strategy", bbox_to_anchor=(1.05, 1), loc="upper left")
	plt.grid(axis="y", linestyle="--", alpha=0.7)
	plt.show()
else:
	print("ℹ️ No non-baseline runs found in the CSV yet. Once you run experiments with pos_debias or self_consist = True, this graph will populate.")

## 3. Verdict Distribution Comparison
Visualizing how debiasing strategies change the model's output distribution (e.g., increasing 'tie' verdicts).

In [ ]:

import glob

# Collect distributions for Qwen Zero-Shot variants
qwen_files = {
    "Baseline": "output/validation_qwen2.5-7b-instruct-cot_baseline.csv",
    "PosDebias": "output/validation_qwen2.5-7b-instruct-cot_pd.csv",
    "SelfConsist": "output/validation_qwen2.5-7b-instruct-cot_sc.csv",
    "PD + SC": "output/validation_qwen2.5-7b-instruct-cot_sc_pd.csv"
}

dist_data = []
for label, path in qwen_files.items():
    if os.path.exists(path):
        v_df = pd.read_csv(path)
        counts = v_df['prediction'].value_counts(normalize=True) * 100
        dist_data.append({
            "Strategy": label,
            "A": counts.get("A", 0),
            "B": counts.get("B", 0),
            "tie": counts.get("tie", 0),
            "neither": counts.get("neither", 0)
        })

dist_df = pd.DataFrame(dist_data).set_index("Strategy")

# Plot stacked bar
ax = dist_df.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#4daf4a', '#377eb8', '#999999', '#e41a1c'])
plt.title("Verdict Distribution: Qwen-2.5-7b CoT Variants")
plt.ylabel("Percentage of Verdicts (%)")
plt.legend(title="Verdict", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()


## 4. Position Bias (B/A Ratio)
Measuring the systematic preference for the second response (Dialog B) vs the first (Dialog A).

In [ ]:

# Calculate B/A ratio
dist_df['B/A Ratio'] = dist_df['B'] / dist_df['A']

plt.figure(figsize=(8, 5))
ax = sns.barplot(x=dist_df.index, y=dist_df['B/A Ratio'], palette="magma")
plt.axhline(1.0, color='red', linestyle='--', label='Perfect Balance (1.0)')
plt.title("Position Bias Analysis (B/A Selection Ratio)")
plt.ylabel("B/A Selection Ratio")
plt.legend()

# Add ratio labels
for i, v in enumerate(dist_df['B/A Ratio']):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

plt.ylim(0, max(dist_df['B/A Ratio']) + 0.2)
plt.show()


## 5. Qwen-2.5-7b Strategy Performance
Directly comparing accuracy across all debiasing and scaling strategies for the Qwen model.

In [ ]:

plt.figure(figsize=(10, 6))
qwen_variants = df[(df["model"] == "qwen2.5-7b-instruct") & (df["prompt_type"] == "zero_shot")]

ax = sns.barplot(
    data=qwen_variants, 
    x="improvement_strategy", 
    y="accuracy_pct",
    palette="viridis",
    order=["Baseline", "PosDebias", "SelfConsist", "PosDebias + SelfConsist"]
)

# Add accuracy labels
for container in ax.containers:
    ax.bar_label(container, fmt='%.1f%%', padding=3)

plt.title("Qwen-2.5-7b Zero-Shot: Impact of Improvement Strategies")
plt.ylabel("Accuracy (%)")
plt.xlabel("Strategy")
plt.ylim(0, max(qwen_variants['accuracy_pct']) + 10)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()


## 6. Qwen CoT Swap Test (Bias Verification)
Comparing the verdict distribution when the order of Dialog A and Dialog B is swapped.

In [ ]:

import pandas as pd
import matplotlib.pyplot as plt

orig_dist = pd.Series({ 'A': 36.0, 'B': 53.0, 'tie': 4.0, 'neither': 7.000000000000001 })
swap_dist = pd.Series({ 'A': 41.0, 'B': 51.0, 'tie': 2.0, 'neither': 6.0 })

plot_df = pd.DataFrame([orig_dist, swap_dist], index=["Original (A, B)", "Swapped (B, A)"])

ax = plot_df.plot(kind='bar', stacked=True, figsize=(10, 6), color=['#4daf4a', '#377eb8', '#999999', '#e41a1c'])
plt.title("Verdict Distribution: Qwen CoT Swap Test")
plt.ylabel("Percentage of Verdicts (%)")
plt.legend(title="Verdict", bbox_to_anchor=(1.05, 1), loc='upper left')

# Add accuracy info in text box
plt.text(1.05, 0.4, f"Original Accuracy: 56.0%\nSwapped Accuracy: 22.0%", 
         transform=ax.transAxes, bbox=dict(facecolor='white', alpha=0.5))

plt.tight_layout()
plt.show()

print(f"Original B/A Ratio: {orig_dist['B'] / (orig_dist['A'] if orig_dist['A'] > 0 else 1):.2f}")
print(f"Swapped B/A Ratio: {swap_dist['B'] / (swap_dist['A'] if swap_dist['A'] > 0 else 1):.2f}")
